# Laboratorium 6 — Projekt generacyjny cz. 1: przygotowanie danych

**Cel:** przygotować dane do modelu Char-RNN — słownik znaków, kodowanie,
okna wejście–cel z przesunięciem o 1 oraz `DataLoader`. To fundament pod
zajęcia 7 (budowa i uczenie modelu) i 8 (generowanie tekstu).

*Prowadzący: dr Tomasz Cieplak / mgr inż. Patryk Marek, Rok akademicki: 2026/27*

## 1. Instalacja i import

In [9]:
!pip -q install torch numpy

In [10]:
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

## 2. Korpus

Domyślnie: mały wbudowany tekst polski (działa offline). Do realnego projektu
użyj większego korpusu — np. klasyki z **Wolnych Lektur** (domena publiczna),
komórka poniżej (opcjonalna).

In [11]:
#ZDANIA = [
#    'Wiatr niósł zapach lasu przez otwarte okno.',
#    'Rzeka płynęła spokojnie wśród zielonych wzgórz.',
#    'Dzieci bawiły się na łące pełnej kwiatów.',
#    'Stary zegar wybił północ w cichym domu.',
#    'Ptaki śpiewały o świcie na gałęziach drzew.',
#    'Deszcz padał miarowo na dachy miasteczka.',
#    'Kot przeciągnął się leniwie w promieniach słońca.',
#    'Podróżnik szedł długą drogą ku odległym górom.',
#]
#tekst = ' '.join(ZDANIA * 40)
#print('Długość tekstu:', len(tekst), 'znaków')

In [12]:
# OPCJONALNIE: realny korpus z Wolnych Lektur (domena publiczna, wymaga sieci)
import urllib.request
url = 'https://wolnelektury.pl/media/book/txt/lalka-tom-pierwszy.txt'
tekst = urllib.request.urlopen(url).read().decode('utf-8')
print('Długość tekstu:', len(tekst))

Długość tekstu: 784082


## 3. Słownik znaków i kodowanie

Budujemy odwzorowania znak ↔ indeks i kodujemy cały tekst na ciąg liczb.

In [13]:
znaki = sorted(set(tekst))
stoi = {z: i for i, z in enumerate(znaki)}
itos = {i: z for z, i in stoi.items()}
V = len(znaki)

def enc(s):  return [stoi[c] for c in s]
def dec(ids): return ''.join(itos[i] for i in ids)

dane = np.array(enc(tekst), dtype=np.int64)
print('Rozmiar słownika (unikalne znaki):', V)
print('Znaki:', repr(''.join(znaki)))
print('Zakodowany tekst, kształt:', dane.shape)

Rozmiar słownika (unikalne znaki): 103
Znaki: "\n\r !'()*,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz«»ÓàçéóöüąćęŁłńŚśŹźŻż–—”„…"
Zakodowany tekst, kształt: (784082,)


## 4. Dataset: okno → cel przesunięty o 1

Każdy przykład to okno `SEKW` znaków (wejście) oraz to samo okno przesunięte
o jeden znak (cel). Model w każdej pozycji przewiduje następny znak.

In [14]:
class DatasetZnakowy(Dataset):
    def __init__(self, dane, sekw):
        self.dane = dane; self.sekw = sekw
    def __len__(self):
        return len(self.dane) - self.sekw
    def __getitem__(self, i):
        x = self.dane[i:i+self.sekw]
        y = self.dane[i+1:i+self.sekw+1]   # przesunięcie o 1
        return torch.tensor(x), torch.tensor(y)

SEKW = 40
ds = DatasetZnakowy(dane, SEKW)
dl = DataLoader(ds, batch_size=32, shuffle=True, drop_last=True)
print('Liczba przykładów:', len(ds))

Liczba przykładów: 784042


## 5. Sanity-check batcha

Zawsze sprawdzamy dane przed treningiem: kształty, dekodowanie i poprawność przesunięcia.

In [15]:
xb, yb = next(iter(dl))
print('Batch X:', xb.shape, ' Y:', yb.shape)
print('Wejście:', repr(dec(xb[0].tolist())))
print('Cel    :', repr(dec(yb[0].tolist())))
assert dec(xb[0].tolist())[1:] == dec(yb[0].tolist())[:-1]
print('Przesunięcie poprawne (cel = wejście + 1).')

Batch X: torch.Size([32, 40])  Y: torch.Size([32, 40])
Wejście: 'sytuacji…\r\n\r\n— Naszego nieszczęśliwego k'
Cel    : 'ytuacji…\r\n\r\n— Naszego nieszczęśliwego kr'
Przesunięcie poprawne (cel = wejście + 1).


## 6. Statystyki zbioru

In [16]:
print(f'Znaków w korpusie : {len(tekst)}')
print(f'Unikalnych znaków : {V}')
print(f'Długość okna SEKW : {SEKW}')
print(f'Liczba przykładów : {len(ds)}')
print(f'Batchy na epokę   : {len(dl)}')

Znaków w korpusie : 784082
Unikalnych znaków : 103
Długość okna SEKW : 40
Liczba przykładów : 784042
Batchy na epokę   : 24501


## Zadanie na sprawozdanie

1. Przygotuj własny zbiór uczący z wybranego tekstu polskiego (np. z Wolnych Lektur).
2. Udokumentuj: rozmiar korpusu, liczbę unikalnych znaków, liczbę przykładów.
3. Pokaż zdekodowany batch i zweryfikuj przesunięcie celu o 1.
4. Zbadaj, jak długość okna `SEKW` wpływa na liczbę przykładów.

Dane przygotowane tutaj wykorzystasz na zajęciach 7 (budowa i uczenie Char-RNN).

Format sprawozdania: `00_materialy_wspolne/szablon_sprawozdania.md`.